In [1]:
import matplotlib
matplotlib.use('Agg')   # ← use file-based rendering instead of screen pop-up
import matplotlib.pyplot as plt
print("✅ Matplotlib backend set to Agg")

✅ Matplotlib backend set to Agg


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
print("✅ Imports done!")

✅ Imports done!


In [3]:
def make_fake_satellite(day, size=128):  # ← changed 256 to 128
    np.random.seed(int(day * 100))
    img = np.zeros((size, size, 3), dtype=np.float32)

    img[:, :, 0] = 0.55 + np.random.normal(0, 0.03, (size, size))
    img[:, :, 1] = 0.45 + np.random.normal(0, 0.03, (size, size))
    img[:, :, 2] = 0.30 + np.random.normal(0, 0.03, (size, size))

    veg = 0.15 + day * 0.25
    img[:size//2, :size//2, 1] += veg
    img[:size//2, :size//2, 0] -= veg * 0.4

    img[size//2:, size//2:, 2] = 0.55 + np.random.normal(0, 0.01, (size//2, size//2))
    img[size//2:, size//2:, 0] = 0.10
    img[size//2:, size//2:, 1] = 0.20

    img[:size//2, size//2:, 0] = 0.60 + np.random.normal(0, 0.02, (size//2, size//2))
    img[:size//2, size//2:, 1] = 0.58 + np.random.normal(0, 0.02, (size//2, size//2))
    img[:size//2, size//2:, 2] = 0.58 + np.random.normal(0, 0.02, (size//2, size//2))

    return np.clip(img, 0, 1)

days   = [i/20.0 for i in range(21)]
images = [make_fake_satellite(d) for d in days]
print(f"✅ Generated {len(images)} images — size: {images[0].shape}")

✅ Generated 21 images — size: (128, 128, 3)


In [4]:
class SatelliteDataset(Dataset):
    def __init__(self, images):
        self.samples = []
        for i in range(len(images) - 2):
            self.samples.append((images[i], images[i+1], images[i+2]))
        print(f"✅ Dataset: {len(self.samples)} triplets")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        before, middle, after = self.samples[idx]
        before_t = torch.tensor(before).permute(2, 0, 1)
        middle_t = torch.tensor(middle).permute(2, 0, 1)
        after_t  = torch.tensor(after).permute(2, 0, 1)
        return before_t, after_t, middle_t

dataset    = SatelliteDataset(images)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)  # ← batch 4→2
print(f"   Batches per epoch: {len(dataloader)}")

✅ Dataset: 19 triplets
   Batches per epoch: 10


In [5]:
class TempoSatModel(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder — smaller filters (32→16, 64→32)
        self.enc1 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),  # ← 32 to 16
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.enc2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # ← 64 to 32
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )

        # Decoder
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.output = nn.Sequential(
            nn.Conv2d(16, 3, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, before, after):
        x  = torch.cat([before, after], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        d1 = self.up1(e2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.output(d1)

model      = TempoSatModel()
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Lighter model built — {total_params:,} parameters")

✅ Lighter model built — 26,147 parameters


In [6]:
device    = torch.device("cpu")
model     = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

EPOCHS = 20  # ← reduced from 30 to 20
print("🚀 Training started...\n")
loss_history = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for before, after, target in dataloader:
        optimizer.zero_grad()
        pred = model(before.to(device), after.to(device))
        loss = criterion(pred, target.to(device))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(dataloader)
    loss_history.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f"   Epoch {epoch+1:2d}/{EPOCHS} — Loss: {avg:.6f}")

print("\n✅ Training complete!")

🚀 Training started...

   Epoch  5/20 — Loss: 0.011810
   Epoch 10/20 — Loss: 0.001312
   Epoch 15/20 — Loss: 0.000954
   Epoch 20/20 — Loss: 0.000866

✅ Training complete!


In [7]:
model.eval()

# ── Step 1: Make prediction ────────────────────────────────────────────────
with torch.no_grad():
    input_before = torch.tensor(make_fake_satellite(0.0)).permute(2,0,1).unsqueeze(0)
    input_after  = torch.tensor(make_fake_satellite(1.0)).permute(2,0,1).unsqueeze(0)
    predicted    = model(input_before, input_after).squeeze(0).permute(1,2,0).numpy()

print("✅ Prediction made successfully!")

✅ Prediction made successfully!


In [8]:
real_day5     = make_fake_satellite(0.5)
baseline_day5 = (make_fake_satellite(0.0) + make_fake_satellite(1.0)) / 2.0

def psnr(pred, real):
    mse = np.mean((pred - real) ** 2)
    return 10 * np.log10(1.0 / mse) if mse > 0 else float('inf')

b_score  = psnr(baseline_day5, real_day5)
ai_score = psnr(predicted, real_day5)

print(f"📊 Dumb baseline : {b_score:.2f} dB")
print(f"   🤖 AI model   : {ai_score:.2f} dB")
print(f"   Improvement   : +{ai_score - b_score:.2f} dB")
print(f"\n{'✅ AI model WON!' if ai_score > b_score else '🔄 Close! Try more epochs.'}")

📊 Dumb baseline : 30.78 dB
   🤖 AI model   : 30.31 dB
   Improvement   : +-0.46 dB

🔄 Close! Try more epochs.


In [ ]:
import os
os.makedirs("../data/results", exist_ok=True)

real_day5     = make_fake_satellite(0.5)
baseline_day5 = (make_fake_satellite(0.0) + make_fake_satellite(1.0)) / 2.0

def psnr(pred, real):
    mse = np.mean((pred - real) ** 2)
    return 10 * np.log10(1.0 / mse) if mse > 0 else float('inf')

b_score  = psnr(baseline_day5, real_day5)
ai_score = psnr(predicted, real_day5)

# Save all 4 images as files instead of displaying in notebook
titles = ["day1_input", "baseline_day5", "ai_predicted_day5", "real_day5"]
imgs   = [make_fake_satellite(0.0), baseline_day5, predicted, real_day5]

for title, img in zip(titles, imgs):
    fig = plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(title, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(f"../data/results/{title}.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print(f"   ✅ Saved: data/results/{title}.png")

print(f"\n📊 Dumb baseline : {b_score:.2f} dB")
print(f"   🤖 AI model   : {ai_score:.2f} dB")
print(f"   Improvement   : +{ai_score - b_score:.2f} dB")
print(f"\n{'✅ AI model WON!' if ai_score > b_score else '🔄 Close! Try more epochs.'}")
print("\n📁 Open your results folder to see the images:")
print("   Documents → TempoSat_Project → data → results")